### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("Why do cats purr?")
response

AIMessage(content='Cats purr for a surprisingly wide range of reasons—most of them are about communication and self‑care, but the mechanics and evolutionary roots are a bit more complex than the simple “I’m happy” stereotype.\n\n| Reason | What’s going on | Why it matters |\n|--------|-----------------|----------------|\n| **Contentment & bonding** | The cat’s laryngeal (voice box) muscles twitch at 25–150\u202fHz, producing a low‑pitched rumble. This is often triggered by gentle stroking, a relaxed posture, or being near a human or another cat. | It signals a calm, safe state and reinforces social bonds. |\n| **Self‑soothing & pain relief** | When a cat is injured, sick, or stressed, it can purr to calm itself. The same 25–150\u202fHz vibrations are thought to stimulate the release of endorphins and reduce pain. | The cat can “treat” itself, much like a human might hum or tap a foot when nervous. |\n| **Healing & bone repair** | Research shows that vibrations in the purring range prom

In [ ]:
from langchain.tools import tool

@tool
def get_exchange_rate(currency_pair:str)->str:
    """Get the exchange rate for a currency pair, e.g. 'USD_EUR'"""
    rates = {
        "USD_EUR": "0.92",
        "USD_GBP": "0.78",
        "EUR_GBP": "0.85",
    }
    return rates.get(currency_pair, f"No exchange rate available for {currency_pair}")


model_with_tools=model.bind_tools([get_exchange_rate])

In [ ]:
response = model_with_tools.invoke("What's the exchange rate for USD to EUR?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

### Tool Execution Loops

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the exchange rate for USD to EUR?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_exchange_rate.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The exchange rate for USD to EUR is 0.92."

In [ ]:
messages

### 🤔 Before you move on
Print the `messages` list from the cell above — it's not just plain strings, it's a mix of typed objects (a human turn, an AI turn carrying tool_calls, a tool result). What exactly are these message types, and how is a `SystemMessage` different from a `HumanMessage`, `AIMessage`, or `ToolMessage`? That's the whole subject of `4-messages.ipynb`.